# Scaling Audio Models Efficiently — Results Analysis
**MSML604/605 | Vyom Agarwal & Mokshda Gangrade**

This notebook loads inference results from `all_inference_results.json` and generates:
- Summary table matching the expected output format
- Compute-Performance Pareto frontier
- Axis sweep plots (audio duration, LoRA rank, unfreeze layers)
- Model size comparison
- Confusion matrices

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── CONFIG — update this path ─────────────────────────────────────────────
RESULTS_JSON = "/scratch/zt1/project/msml605/user/mokshdag/inference_results/all_inference_results.json"
OUTPUT_DIR   = "/scratch/zt1/project/msml605/user/mokshdag/inference_results/plots/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Imports OK')

In [ ]:
# ── Load results ──────────────────────────────────────────────────────────
with open(RESULTS_JSON) as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} runs")
df.head()

In [ ]:
# ── SUMMARY TABLE (matches expected output format) ─────────────────────────
summary_cols = [
    'model', 'audio_len', 'lora_r', 'adaptation',
    'flops_b', 'trainable_params_m', 'ua_pct', 'rtf', 'gpu_name'
]

display_df = df[summary_cols].copy()
display_df.columns = [
    'Model', 'Audio(s)', 'Rank', 'Adaptation',
    'FLOPs(B)', 'Trainable Params(M)', 'UA(%)', 'RTF', 'GPU'
]
display_df = display_df.sort_values('UA(%)', ascending=False)
display_df['UA(%)']             = display_df['UA(%)'].round(2)
display_df['FLOPs(B)']          = display_df['FLOPs(B)'].round(1)
display_df['Trainable Params(M)'] = display_df['Trainable Params(M)'].round(1)
display_df['RTF']               = display_df['RTF'].round(5)

print('=== RESULTS TABLE ===')
display_df

In [ ]:
# ── STYLED TABLE (matching paper format) ──────────────────────────────────
def style_table(df):
    return df.style\
        .background_gradient(subset=['UA(%)'], cmap='Greens')\
        .background_gradient(subset=['FLOPs(B)'], cmap='Reds_r')\
        .background_gradient(subset=['RTF'], cmap='Reds_r')\
        .set_properties(**{'text-align': 'center', 'font-size': '12px'})\
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#2d4059'),
                ('color', 'white'),
                ('font-weight', 'bold'),
                ('text-align', 'center'),
                ('padding', '8px'),
            ]},
            {'selector': 'td', 'props': [('padding', '6px 12px')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f0f4f8')]},
        ])

style_table(display_df)

In [ ]:
# ── PARETO FRONTIER ───────────────────────────────────────────────────────
def compute_pareto(df, x_col, y_col, minimize_x=True, maximize_y=True):
    pts = df.to_dict('records')
    pareto = []
    for p in pts:
        dominated = False
        for q in pts:
            if p == q: continue
            xb = q[x_col] <= p[x_col] if minimize_x else q[x_col] >= p[x_col]
            yb = q[y_col] >= p[y_col] if maximize_y else q[y_col] <= p[y_col]
            xs = q[x_col] < p[x_col] if minimize_x else q[x_col] > p[x_col]
            ys = q[y_col] > p[y_col] if maximize_y else q[y_col] < p[y_col]
            if xb and yb and (xs or ys):
                dominated = True
                break
        if not dominated:
            pareto.append(p)
    return pd.DataFrame(sorted(pareto, key=lambda x: x[x_col]))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: FLOPs vs UA ─────────────────────────────────────────────────────
ax = axes[0]
color_map = {
    'Full': '#27ae60',
    'LoRA-only': '#e74c3c',
    'LoRA+top4': '#2d4059',
    'LoRA+top8': '#8e44ad',
}
for _, row in df.iterrows():
    color = color_map.get(row['adaptation'], '#95a5a6')
    ax.scatter(row['flops_b'], row['ua_pct'], color=color, s=150,
               edgecolors='black', lw=0.5, zorder=5)
    ax.annotate(row['model'][:15] + f"\nr={row['lora_r']}, {row['audio_len']}s",
                (row['flops_b'], row['ua_pct']),
                textcoords='offset points', xytext=(6, 4), fontsize=7)

# Pareto frontier
valid = df[df['flops_b'].notna() & df['ua_pct'].notna()]
if len(valid) >= 2:
    pareto_df = compute_pareto(valid, 'flops_b', 'ua_pct')
    ax.plot(pareto_df['flops_b'], pareto_df['ua_pct'],
            'k--', lw=1.5, label='Pareto frontier', zorder=4)
    ax.scatter(pareto_df['flops_b'], pareto_df['ua_pct'],
               color='gold', s=300, zorder=6, edgecolors='black', lw=1.5,
               marker='*', label='Pareto optimal')

patches = [mpatches.Patch(color=c, label=k) for k, c in color_map.items()]
ax.legend(handles=patches + [plt.Line2D([0],[0],ls='--',color='k',label='Pareto frontier'),
                              plt.scatter([],[],marker='*',color='gold',s=200,label='Pareto optimal')],
          fontsize=8, loc='lower right')
ax.set_xlabel('FLOPs (B) — lower = cheaper inference', fontsize=12)
ax.set_ylabel('UA (%) — higher = better', fontsize=12)
ax.set_title('Compute-Performance Pareto Frontier\n(FLOPs vs UA)', fontsize=13)
ax.grid(True, alpha=0.3)

# ── Right: RTF vs UA ──────────────────────────────────────────────────────
ax = axes[1]
for _, row in df.iterrows():
    if pd.isna(row.get('rtf')): continue
    color = color_map.get(row['adaptation'], '#95a5a6')
    ax.scatter(row['rtf'], row['ua_pct'], color=color, s=150,
               edgecolors='black', lw=0.5, zorder=5)
    ax.annotate(row['model'][:12],
                (row['rtf'], row['ua_pct']),
                textcoords='offset points', xytext=(5, 4), fontsize=7)

rtf_valid = df[df['rtf'].notna() & df['ua_pct'].notna()]
if len(rtf_valid) >= 2:
    pareto_rtf = compute_pareto(rtf_valid, 'rtf', 'ua_pct')
    ax.plot(pareto_rtf['rtf'], pareto_rtf['ua_pct'], 'k--', lw=1.5)
    ax.scatter(pareto_rtf['rtf'], pareto_rtf['ua_pct'],
               color='gold', s=300, zorder=6, edgecolors='black', lw=1.5, marker='*')

ax.set_xlabel('RTF — lower = faster', fontsize=12)
ax.set_ylabel('UA (%) — higher = better', fontsize=12)
ax.set_title('Compute-Performance Pareto Frontier\n(RTF vs UA)', fontsize=13)
ax.grid(True, alpha=0.3)

plt.suptitle('CREMA-D Emotion Recognition — Pareto Frontier', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pareto_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pareto_frontier.png')

In [ ]:
# ── AXIS SWEEP PLOTS ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

base_model = 'wav2vec2-large-robust'

# xT: Audio duration
ax = axes[0]
audio_df = df[
    df['model'].str.contains('large-robust') &
    (df['mode'] == 'lora') &
    (df['lora_r'] == 16) &
    df['audio_len'].isin([2.0, 4.0, 6.0])
].sort_values('audio_len')
if len(audio_df):
    ax.plot(audio_df['audio_len'], audio_df['ua_pct'], 'o-', color='#2d4059', lw=2.5, ms=12)
    for _, r in audio_df.iterrows():
        ax.annotate(f"{r['ua_pct']:.1f}%", (r['audio_len'], r['ua_pct']),
                    textcoords='offset points', xytext=(5, 8), fontsize=11, fontweight='bold')
ax.set_xlabel('Max Audio Length (seconds)', fontsize=12)
ax.set_ylabel('UA (%)', fontsize=12)
ax.set_title('xT Axis: Audio Duration vs UA', fontsize=13)
ax.set_xticks([2, 4, 6])
ax.grid(True, alpha=0.3)
ax.set_ylim(40, 75)

# LoRA rank
ax = axes[1]
rank_df = df[
    df['model'].str.contains('large-robust') &
    (df['mode'] == 'lora') &
    (df['audio_len'] == 4.0) &
    df['lora_r'].notna()
].sort_values('lora_r')
if len(rank_df):
    ax.plot(rank_df['lora_r'], rank_df['ua_pct'], 's-', color='#e74c3c', lw=2.5, ms=12)
    for _, r in rank_df.iterrows():
        ax.annotate(f"{r['ua_pct']:.1f}%", (r['lora_r'], r['ua_pct']),
                    textcoords='offset points', xytext=(5, 8), fontsize=11, fontweight='bold')
    # Also plot trainable params on secondary axis
    ax2 = ax.twinx()
    ax2.plot(rank_df['lora_r'], rank_df['trainable_params_m'],
             '^--', color='#f39c12', lw=1.5, ms=8, alpha=0.7, label='Trainable params')
    ax2.set_ylabel('Trainable Params (M)', fontsize=10, color='#f39c12')
    ax2.tick_params(axis='y', labelcolor='#f39c12')
ax.set_xlabel('LoRA Rank (r)', fontsize=12)
ax.set_ylabel('UA (%)', fontsize=12)
ax.set_title('LoRA Rank vs UA (+ Params)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_ylim(40, 80)

# Unfreeze layers (DAMA)
ax = axes[2]
unfreeze_df = df[
    df['model'].str.contains('large-robust') &
    (df['mode'] == 'lora') &
    (df['audio_len'] == 4.0) &
    (df['lora_r'] == 16)
].sort_values('unfreeze_layers')
if len(unfreeze_df):
    ax.plot(unfreeze_df['unfreeze_layers'], unfreeze_df['ua_pct'],
            '^-', color='#27ae60', lw=2.5, ms=12)
    for _, r in unfreeze_df.iterrows():
        ax.annotate(f"{r['ua_pct']:.1f}%", (r['unfreeze_layers'], r['ua_pct']),
                    textcoords='offset points', xytext=(5, 8), fontsize=11, fontweight='bold')
ax.set_xlabel('Unfrozen Top Encoder Layers', fontsize=12)
ax.set_ylabel('UA (%)', fontsize=12)
ax.set_title('DAMA: Unfreeze Layers vs UA', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_ylim(35, 75)

plt.suptitle('CREMA-D Emotion Recognition — Axis Sweeps', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/axis_sweeps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved axis_sweeps.png')

In [ ]:
# ── MODEL SIZE COMPARISON ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

plot_df = df.sort_values('ua_pct', ascending=False)
short_names = [
    r['run_name'].replace('ser-', '').replace('-cremad', '').replace('lora-', '')
    for _, r in plot_df.iterrows()
]
colors = [
    '#27ae60' if 'full' in r['adaptation'].lower() else
    '#e74c3c' if 'base' in r['model'] else
    '#8e44ad' if 'whisper' in r['model'] else
    '#2d4059'
    for _, r in plot_df.iterrows()
]

bars = ax.bar(range(len(plot_df)), plot_df['ua_pct'], color=colors,
              edgecolor='black', lw=0.5)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('UA (%)', fontsize=12)
ax.set_title('All Configurations — UA Comparison\nCREMA-D 6-class Emotion Recognition', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 95)
ax.axhline(y=100/6, color='gray', ls=':', lw=1.5, label='Random baseline (16.7%)')

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{row['ua_pct']:.1f}%\n{row['trainable_params_m']:.0f}M",
            ha='center', fontsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#27ae60', label='Full finetune'),
    Patch(color='#e74c3c', label='wav2vec2-base LoRA'),
    Patch(color='#2d4059', label='wav2vec2-large-robust LoRA'),
    plt.Line2D([0],[0],ls=':',color='gray',label='Random baseline'),
], fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_comparison.png')

In [ ]:
# ── CONFUSION MATRIX for best model ───────────────────────────────────────
best_run = df.sort_values('ua_pct', ascending=False).iloc[0]
print(f"Best run: {best_run['run_name']} | UA={best_run['ua_pct']:.2f}%")

cm = np.array(best_run['confusion_matrix'])
label_names = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust']

fig, ax = plt.subplots(figsize=(8, 7))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix (Normalized)\n{best_run["run_name"]}\nUA={best_run["ua_pct"]:.2f}%', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix_best.png')

In [ ]:
# ── EFFICIENCY METRIC: UA / FLOPs ─────────────────────────────────────────
df['efficiency'] = df['ua_pct'] / df['flops_b'].clip(lower=0.01)

fig, ax = plt.subplots(figsize=(12, 6))
eff_df = df.sort_values('efficiency', ascending=False)
short = [r['run_name'].replace('ser-','').replace('-cremad','')[:35] for _, r in eff_df.iterrows()]
colors_eff = ['#27ae60' if 'full' in r['adaptation'].lower() else
               '#e74c3c' if 'base' in r['model'] else '#2d4059'
               for _, r in eff_df.iterrows()]

bars = ax.bar(range(len(eff_df)), eff_df['efficiency'], color=colors_eff,
              edgecolor='black', lw=0.5)
ax.set_xticks(range(len(eff_df)))
ax.set_xticklabels(short, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Efficiency = UA(%) / FLOPs(B)', fontsize=12)
ax.set_title('Efficiency Score — UA per Unit FLOPs\n(Higher = Better Compute-Performance Tradeoff)', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')
for bar, (_, row) in zip(bars, eff_df.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{row['efficiency']:.2f}", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/efficiency_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved efficiency_score.png')

In [ ]:
# ── PRINT FINAL TABLE (paper format) ─────────────────────────────────────
print('\n' + '='*110)
print(f'{"Model":<25} {"Audio":>6} {"Rank":>5} {"Adaptation":<12} {"FLOPs(B)":>9} {"Params(M)":>10} {"UA%":>7} {"RTF":>8} {"GPU":<20}')
print('='*110)
for _, r in df.sort_values('ua_pct', ascending=False).iterrows():
    rank = str(int(r['lora_r'])) if pd.notna(r.get('lora_r')) else '—'
    rtf  = f"{r['rtf']:.5f}" if pd.notna(r.get('rtf')) else 'N/A'
    print(
        f"{str(r['model'])[:25]:<25} "
        f"{r['audio_len']:>6} "
        f"{rank:>5} "
        f"{r['adaptation']:<12} "
        f"{r['flops_b']:>9.1f} "
        f"{r['trainable_params_m']:>10.1f} "
        f"{r['ua_pct']:>7.2f} "
        f"{rtf:>8} "
        f"{str(r['gpu_name'])[:20]:<20}"
    )
print('='*110)